In [0]:
from pyspark.sql import functions as F

orders = spark.table("project.bronze.orders")
cust   = spark.table("project.bronze.customers")

enriched = (orders.alias("o")
  .join(cust.alias("c"), "customer_id", "left")
  .select(
      "o.order_id",
      "o.customer_id",
      "o.status",
      "o.order_ts",
      "o.ingest_date",
      "c.customer_name",
      "c.city",
      "c.segment",
      F.current_timestamp().alias("silver_ingest_ts")
  ))

(enriched.write.format("delta")
 .mode("overwrite")
 .option("overwriteSchema", "true")
 .saveAsTable("project.silver.orders_enriched"))

dbutils.jobs.taskValues.set("silver_rows", enriched.count())
print("Heavy transform completed. Wrote project.silver.orders_enriched")

display(enriched.orderBy("order_id"))
